# Status Report 2024/10/23
## Progress
1. Basic Understanding of TLD IoT Hamburg / OGC SensorThingsAPI
2. Crude Implementation of a DataFetcher class
3.

## Problems
1. Connections Times
2. observedArea vs. Location
3. General Lack of traffic light data

## Goals
1. Implement method for finding ID by coords
2. Implement local processing of data 
3. Implement methods for handling Ingress/Egress lanes

## Resources
1. 

## Licenses
1. OGC SensorThingsAPI is License Free

![Couldn't find Image](./ER-Diagramm_SensorThingsAPI.png)

In [9]:
from DataFetcher import DataFetcher
import json

In [10]:
data_fetcher = DataFetcher()

endpoint = "Datastreams"
params = {
    "$filter": "properties/layerName eq 'primary_signal'",
    "$expand" : "Thing($expand=Locations)",
}

data_fetcher.fetch_data(endpoint,params)

print(f"Retrieved {len(data_fetcher.get_data())} datastreams.")

Retrieved 19969 datastreams.


In [12]:
print(json.dumps(data_fetcher.get_data()[1],indent=4))

{
    "@iot.selfLink": "https://tld.iot.hamburg.de/v1.1/Datastreams(11325)",
    "@iot.id": 11325,
    "name": "Primary signal heads at 2382_6",
    "description": "Datastream to broadcast the lane connection's signal value of a signal group",
    "observationType": "http://www.opengis.net/def/ogc-om/OM_CategoryObservation",
    "unitOfMeasurement": {
        "name": "Signal Status ",
        "symbol": "[0-6,9]",
        "definition": "0=dark,1=red,2=amber,3=green,4=red-amber,5=amber-flashing,6=green-flashing,9=unknown"
    },
    "observedArea": {
        "type": "LineString",
        "coordinates": [
            [
                9.9097591,
                53.6044949
            ],
            [
                9.9097595,
                53.6044955
            ]
        ]
    },
    "phenomenonTime": "2021-09-16T14:57:53Z/2024-10-23T12:22:23Z",
    "properties": {
        "metadata": "https://registry.gdi-de.org/id/de.hh/fa5588ec-bc36-4070-8871-6bf69365fe79",
        "layerName": "pr

In [ ]:
import folium
from IPython.display import display

def switch_coords(coords):
    temp = coords[0]
    coords[0] = coords[1]
    coords[1] = temp

#save possible colors for markers
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black']
def create_marker(coords,color_index):
    folium.CircleMarker(location=coords,radius=1,color = colors[color_index], fill = True, fill_color = colors[color_index], opacity = 1).add_to(map_object)
#define starting point for map (chosen arbitrarily)
hamburg_mid_point = [53.571974, 9.969119]

#create maps
map_object = folium.Map(location=hamburg_mid_point, zoom_start=12)


counter = 0
for stream in data_fetcher.get_data():
    if "observedArea" not in stream.keys():
        continue
    
    if stream["observedArea"]["type"] == "Point":
        coords = stream["observedArea"]["coordinates"]
        switch_coords(coords)
        create_marker(coords,0)
    elif stream["observedArea"]["type"] == "LineString":
        coord_pairs = stream["observedArea"]["coordinates"]
        for coords in coord_pairs:
            switch_coords(coords)  # latidute and longitude are given switch up by the API
            #add Marker for each coordinate_pair
            create_marker(coords,1)
    else:
        coord_pairs = stream["observedArea"]["coordinates"][0]
        for coords in coord_pairs:
            switch_coords(coords)  # latidute and longitude are given switch up by the API
            create_marker(coords,2)

# Display the map
display(map_object)